In [253]:
import os
import time
import random
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors
from unittest.mock import MagicMock
from dataclasses import dataclass, field



In [254]:
load_dotenv()
client = genai.Client()

### Gemini api call with streaming enabled

In [255]:
@dataclass
class API_Configs:
    model_config = types.GenerateContentConfig(system_instruction="do not provide code",max_output_tokens=4096)
    
    model_tiers = {
    "high": {"model_name": "gemini-3.5-flash"},
    "medium": {"model_name": "gemini-2.5-flash"},
    "low": {"model_name": "gemini-3.1-flash-lite"}
}

In [256]:
@dataclass
class Retry_Config:
    max_attempts: int = 3
    base_delay: float = 1.0
    max_delay: float = 30.0

    retry_status_codes = (429, 500, 503, 504)

In [257]:
model_priority = "high"

In [258]:
tier = API_Configs.model_tiers.get(model_priority.lower(), API_Configs.model_tiers["medium"])
model_name = tier["model_name"]

### api call with retry error handling, exponential backoff, and streaming

In [259]:
def execute_prompt(model_name, prompt):
    for attempt in range (1, Retry_Config.max_attempts + 1):
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            else:
                print(f"Sending initial request...")

            stream = client.models.generate_content_stream(model = model_name, contents = prompt)

            for chunk in stream:
                yield chunk

            return 
        
        except errors.APIError as e:
            if e.code in Retry_Config.retry_status_codes:
                if attempt == Retry_Config.max_attempts:
                    print("Max attempts reached.")
                    raise e
            
                new_delay = random.uniform(0.5, 1.5) * Retry_Config.base_delay * (2 ** (attempt - 1))
                clamped_delay = min(new_delay, Retry_Config.max_delay)

                print(f" Code {e.code}. Backing off. Waiting {clamped_delay:.2f}s...")
                time.sleep(clamped_delay)
                continue

            else:
                status_reason = getattr(e, "status", "UNKNOWN")
                
                if e.code == 404 or status_reason == "NOT_FOUND":
                    print(f"Model Not Found [404]: '{model_name}' does not exist or is deprecated.")
                    
                elif e.code in (401, 403) or status_reason == "PERMISSION_DENIED":
                    print(f"Authentication Failed [{e.code}]: Check your API key, project quotas, or permissions.")
                    
                elif e.code == 400 or status_reason == "INVALID_ARGUMENT":
                    print(f"Invalid Argument [400]: Malformed structure or prompt content limits exceeded.")
                    
                else:
                    print(f"Non-retryable error [{status_reason}] ({e.code}): {e.message}")
                    
                raise e

In [260]:
response_stream = execute_prompt(model_name = model_name, prompt = "Say hello in 10 random languages")

for chunk in response_stream:
    print(chunk.text, end="", flush=True)

Sending initial request...
Here is "hello" in 10 different languages from around the world:

1. **Japanese**: こんにちは (Konnichiwa)
2. **Swahili**: Jambo
3. **German**: Hallo
4. **Hindi**: नमस्ते (Namaste)
5. **Arabic**: مرحبا (Marhaban)
6. **Portuguese**: Olá
7. **Finnish**: Hei
8. **Maori**: Kia ora
9. **Russian**: Привет (Privet)
10. **Hawaiian**: Aloha